# charmnumeric quickstart

charmnumeric is a NumPy-like array library whose arrays live on a Charm++ server.
Python records operations lazily and sends them to the server; values come back
only when you look at them (display, `print`, `float()`, `np.asarray`, plotting).

In [ ]:
import numpy as np
import charmnumeric as cnp

# Start a server in a terminal, then connect to it:
#   charmrun +p4 src/build/server.out ++server ++server-port 1234 ++local
# Once auto-start (#11) is merged, `cnp.start()` replaces this cell.
import os
cnp.CharmNumericInterface().connect(os.environ.get("CNUM_HOST", "127.0.0.1"),
                                    int(os.environ.get("CNUM_PORT", "1234")), 4)

## Arrays and lazy evaluation

Creating arrays and doing arithmetic only records work. At the end of each cell
the recorded work is sent to the server, which computes while you write the next
cell. Displaying an array waits for its values; large arrays show only the corners
NumPy would print, and only those are fetched.

In [ ]:
x = cnp.linspace(0, 8 * np.pi, 1_000_001)
y = cnp.exp(-x / 8) * cnp.sin(x)
y

In [ ]:
print(y.shape, y.dtype, y.nbytes)
print("max:", float(cnp.max(y)), " mean:", float(cnp.mean(y)))

`repr(y)` never contacts the server (useful in debuggers and logs); `print(y)` and
the notebook display fetch a bounded preview.

In [ ]:
repr(y)

## Timing

Because work is asynchronous, time it with `block_until_ready()` (or
`cnp.synchronize()`), which waits until the server has finished everything recorded so far.

In [ ]:
import time

t = time.perf_counter()
energy = cnp.sum(y * y)
energy.block_until_ready()
print(f"sum(y*y) = {float(energy):.6f} in {1e3 * (time.perf_counter() - t):.1f} ms")

## Plotting

Matplotlib converts arrays with one fetch each (`np.asarray(a)` or `a.to_numpy()`).
Slice before plotting large arrays so only what is drawn is transferred.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 2.5))
ax.plot(x[::1000], y[::1000])
ax.set_xlabel("x")
ax.set_ylabel("y");

In [ ]:
n = 128
u = cnp.zeros((n, n))
u[0, :] = 1.0
for _ in range(100):
    u[1:-1, 1:-1] = 0.25 * (u[:-2, 1:-1] + u[2:, 1:-1] + u[1:-1, :-2] + u[1:-1, 2:])
plt.imshow(u, cmap="inferno")
plt.colorbar();

## Display options

`cnp.set_display(preview=False)` shows only shape, dtype and size (no fetch);
`threshold`/`edgeitems` work like `numpy.set_printoptions` and bound the fetch.

In [ ]:
cnp.set_display(preview=False)
u

In [ ]:
cnp.set_display(preview=True, edgeitems=2)
u

In [ ]:
cnp.set_display(edgeitems=3)
host = np.asarray(u[:4, :4])  # a NumPy copy
type(host), host.round(3)

## Interrupts, re-running cells, restarts

* *Interrupt the kernel* (or Ctrl-C) stops waiting for a long result and leaves the
  session usable; work already sent still runs on the server, so the next fetch
  waits for it.
* Re-running the startup cell reconnects to the same server and keeps your arrays.
* A server serves one kernel process. With a server you started yourself, *Restart
  kernel* also needs a server restart (connecting raises `ConnectionError` otherwise);
  `cnp.start()` gives each kernel its own server, so a restart just works.
* Set `CNUM_FLUSH_AFTER_CELL=0` to keep work queued across cells instead of sending
  it when each cell ends.